In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: /home/sreethanu/lidc_ldri_dataset_1/lung_cancer_vl_jepa_lidc


In [3]:
import warnings
warnings.filterwarnings("ignore")
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

from src.models.encoder_3d import ViT3DEncoder
from src.models.classification_head import ClassificationHead
from src.utils.config import Config
from src.utils.seed import set_global_seed
from src.utils.metrics import compute_classification_metrics

In [4]:
config_path = PROJECT_ROOT / "configs" / "jepa_config.yaml"
config = Config(str(config_path))

set_global_seed(
    config.get("project.seed"),
    deterministic=config.get("project.deterministic")
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Setting global seed: 42
Enabling deterministic mode (may reduce performance)
Reproducibility configured successfully.
Device: cuda


In [5]:
metadata_path = PROJECT_ROOT / "data" / "processed" / "metadata" / "labels.csv"
metadata = pd.read_csv(metadata_path)

print(metadata["label"].value_counts())


label
0    475
1    408
Name: count, dtype: int64


In [6]:
from torch.utils.data import Dataset
import numpy as np

class LIDCClassificationDataset(Dataset):
    def __init__(self, processed_dir, metadata):
        self.processed_dir = Path(processed_dir)
        self.metadata = metadata.reset_index(drop=True)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        series_uid = row["series_uid"]
        label = row["label"]

        series_path = self.processed_dir / "patches" / series_uid
        patch_files = list(series_path.glob("*.npz"))

        patch_file = patch_files[np.random.randint(len(patch_files))]
        data = np.load(patch_file)

        volume = torch.tensor(data["patch"], dtype=torch.float32)

        if volume.ndim == 3:
            volume = volume.unsqueeze(0)

        return volume, torch.tensor(label, dtype=torch.long)


In [7]:
dataset = LIDCClassificationDataset(
    processed_dir=PROJECT_ROOT / "data" / "processed",
    metadata=metadata
)

val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))


Train size: 707
Val size: 176


In [8]:
online_encoder = ViT3DEncoder(
    input_size=tuple(config.get("data.input_shape")[1:]),
    patch_size=tuple(config.get("model.encoder.patch_size")),
    embed_dim=config.get("model.encoder.embed_dim"),
    depth=config.get("model.encoder.depth"),
    num_heads=config.get("model.encoder.num_heads"),
).to(device)

checkpoint_path = PROJECT_ROOT / "checkpoints" / "pretraining" / "jepa_epoch_100.pth"
checkpoint = torch.load(checkpoint_path, map_location=device)

online_encoder.load_state_dict(checkpoint["online_encoder"])

print("Pretrained encoder loaded.")


Pretrained encoder loaded.


In [9]:
classifier = ClassificationHead(
    embed_dim=config.get("model.encoder.embed_dim"),
    hidden_dims=config.get("model.classifier.hidden_dims"),
    num_classes=2
).to(device)

model = torch.nn.Sequential(
    online_encoder,
    classifier
).to(device)


In [10]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

criterion = torch.nn.CrossEntropyLoss()


In [11]:
epochs = 30
best_auc = 0

save_dir = PROJECT_ROOT / "checkpoints" / "finetune"
save_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for volumes, labels in train_loader:

        volumes = volumes.to(device)
        labels = labels.to(device)

        logits = model(volumes)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f}")


Epoch 1 | Train Loss: 0.6983
Epoch 2 | Train Loss: 0.6926
Epoch 3 | Train Loss: 0.6974
Epoch 4 | Train Loss: 0.6952
Epoch 5 | Train Loss: 0.6938
Epoch 6 | Train Loss: 0.6920
Epoch 7 | Train Loss: 0.6926
Epoch 8 | Train Loss: 0.6909
Epoch 9 | Train Loss: 0.6970
Epoch 10 | Train Loss: 0.6933
Epoch 11 | Train Loss: 0.6907
Epoch 12 | Train Loss: 0.6878
Epoch 13 | Train Loss: 0.6895
Epoch 14 | Train Loss: 0.6928
Epoch 15 | Train Loss: 0.6940
Epoch 16 | Train Loss: 0.6938
Epoch 17 | Train Loss: 0.6955
Epoch 18 | Train Loss: 0.6895
Epoch 19 | Train Loss: 0.6881
Epoch 20 | Train Loss: 0.6923
Epoch 21 | Train Loss: 0.6901
Epoch 22 | Train Loss: 0.6892
Epoch 23 | Train Loss: 0.6913
Epoch 24 | Train Loss: 0.6890
Epoch 25 | Train Loss: 0.6911
Epoch 26 | Train Loss: 0.6900
Epoch 27 | Train Loss: 0.6912
Epoch 28 | Train Loss: 0.6879
Epoch 29 | Train Loss: 0.6898
Epoch 30 | Train Loss: 0.6889


In [12]:
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for volumes, labels in val_loader:

        volumes = volumes.to(device)

        logits = model(volumes)
        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

metrics = compute_classification_metrics(
    np.array(all_labels),
    np.array(all_preds),
    np.array(all_probs)
)

print("Fine-Tuning Metrics:")
print(metrics)


Fine-Tuning Metrics:
{'accuracy': 0.5056818181818182, 'balanced_accuracy': 0.5, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'sensitivity': np.float64(0.0), 'specificity': np.float64(1.0), 'ppv': 0.0, 'npv': np.float64(0.5056818181818182), 'mcc': 0.0, 'roc_auc': 0.5292522278186749, 'pr_auc': 0.5587317050520789}


In [13]:
save_dir = PROJECT_ROOT / "checkpoints" / "finetune"
save_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model": model.state_dict(),
    },
    save_dir / "final_model.pth"
)

print("Final fine-tuned model saved.")


Final fine-tuned model saved.
